# 11e. Final DML Regression

**Role of this notebook.** This notebook reports the final causal estimate for the paper. It combines the debiased outcome from notebook 11b with the debiased treatment from notebook 11d, then estimates the final DML regression on the held-out test users.

The outcome is the residualized change in unrestricted choice threshold:

$$
\widetilde{\Delta \tau}_{it}
=
\Delta \tau_{it}-\widehat{\mathbb{E}}[\Delta \tau_{it}\mid X_{it}].
$$

The treatment variables are the actual-minus-toy recommendation shocks:

$$
D^\mu_{it}=\mu^{real}_{i,t+1}-\mu^{toy}_{i,t+1},
$$

$$
D^\sigma_{it}=\sigma^{real}_{i,t+1}-\sigma^{toy}_{i,t+1}.
$$

The final regression is:

$$
\widetilde{\Delta \tau}_{it}
=
\alpha
+
\beta_\mu D^\mu_{it}
+
\beta_\sigma D^\sigma_{it}
+
\varepsilon_{it}.
$$

Standard errors are clustered by `user_id` because each test user can contribute multiple adjacent-session transitions.


## Main Estimation Sample

The preferred specification uses two pre-specified reliability filters:

1. **Tau-bound filter.** Drop transitions where either session-level unrestricted tau estimate is at the optimizer boundary.
2. **Treatment-outlier filter.** Within the no-bound sample, trim observations outside the 1st and 99th percentiles of both treatment variables:

$$
q_{0.01}(D^\mu) \le D^\mu_{it} \le q_{0.99}(D^\mu),
$$

$$
q_{0.01}(D^\sigma) \le D^\sigma_{it} \le q_{0.99}(D^\sigma).
$$

The main result does **not** trim the outcome. Outcome trimming can change the target estimand and is less defensible than trimming treatment outliers. A small outcome-winsorized robustness check is reported separately.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import statsmodels.api as sm

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

ROOT = Path('/Users/haozhangao/Desktop/RecSys Research')
OUT_DIR = ROOT / 'python_code_new' / 'outputs'

TREATMENT_PATH = OUT_DIR / 'causal_debiased_treatment_transitions.parquet'
TAU_SESSIONS_PATH = OUT_DIR / 'causal_tau_choice_sessions.parquet'
RESULTS_CSV = OUT_DIR / 'causal_final_dml_regression_results.csv'
RESULTS_JSON = OUT_DIR / 'causal_final_dml_regression_results.json'
REPORT_TABLE_CSV = OUT_DIR / 'causal_final_dml_report_table.csv'

for path in [TREATMENT_PATH, TAU_SESSIONS_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)


## 1. Load Final-Stage Data

Notebook 11d stores one row per test transition. The required variables are `delta_tau_resid`, `D_mu`, `D_sigma`, and `user_id`.


In [2]:
treat = pd.read_parquet(TREATMENT_PATH)
required_cols = [
    'user_id', 'user_idx_int', 'burst_id', 'session', 'next_session',
    'delta_tau', 'pred_delta_tau', 'delta_tau_resid', 'D_mu', 'D_sigma'
]
missing = [c for c in required_cols if c not in treat.columns]
if missing:
    raise KeyError(f'Missing required columns from 11d output: {missing}')

reg = (
    treat[required_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=['user_id', 'delta_tau_resid', 'D_mu', 'D_sigma'])
    .copy()
)
reg['D_norm'] = np.sqrt(reg['D_mu'] ** 2 + reg['D_sigma'] ** 2)

print('test transitions:', f'{len(reg):,}')
print('test users:', f"{reg['user_id'].nunique():,}")
display(reg[['delta_tau_resid', 'D_mu', 'D_sigma', 'D_norm']].describe(percentiles=[0.01, 0.5, 0.99]).T)


test transitions: 4,069
test users: 356


,count,mean,std,min,1%,50%,99%,max
delta_tau_resid,"4,069.000000",0.088446,3.838854,-14.635182,-9.609496,0.131761,9.207921,16.925554
D_mu,"4,069.000000",0.017888,0.026709,-0.092096,-0.021062,0.009424,0.116909,0.303569
D_sigma,"4,069.000000",-0.013095,0.021416,-0.197889,-0.096014,-0.005597,0.013062,0.085790
D_norm,"4,069.000000",0.025984,0.031437,0.000048,0.000565,0.014723,0.145742,0.362373


## 2. Construct The Preferred Sample

The tau-bound flag is merged from notebook 11a. Treatment trimming percentiles are computed after dropping tau-bound transitions.


In [3]:
tau_sessions = pd.read_parquet(
    TAU_SESSIONS_PATH,
    columns=['user_id', 'burst_id', 'session', 'tau_at_bound']
)

tau_t = tau_sessions.rename(columns={'tau_at_bound': 'tau_at_bound_t'})
tau_t1 = tau_sessions.rename(columns={'session': 'next_session', 'tau_at_bound': 'tau_at_bound_t1'})

reg = reg.merge(tau_t, on=['user_id', 'burst_id', 'session'], how='left', validate='many_to_one')
reg = reg.merge(tau_t1, on=['user_id', 'burst_id', 'next_session'], how='left', validate='many_to_one')
reg['no_tau_bound'] = (~reg['tau_at_bound_t'].fillna(True)) & (~reg['tau_at_bound_t1'].fillna(True))

no_bound = reg[reg['no_tau_bound']].copy()
trim_quantiles = no_bound[['D_mu', 'D_sigma']].quantile([0.01, 0.99])
D_mu_lo, D_mu_hi = float(trim_quantiles.loc[0.01, 'D_mu']), float(trim_quantiles.loc[0.99, 'D_mu'])
D_sigma_lo, D_sigma_hi = float(trim_quantiles.loc[0.01, 'D_sigma']), float(trim_quantiles.loc[0.99, 'D_sigma'])

preferred_mask = (
    reg['no_tau_bound']
    & reg['D_mu'].between(D_mu_lo, D_mu_hi)
    & reg['D_sigma'].between(D_sigma_lo, D_sigma_hi)
)
preferred = reg[preferred_mask].copy()

sample_summary = pd.DataFrame([
    {'sample': 'all_test', 'rows': len(reg), 'users': reg['user_id'].nunique()},
    {'sample': 'no_tau_bound', 'rows': len(no_bound), 'users': no_bound['user_id'].nunique()},
    {'sample': 'preferred_no_bound_trim_D_1_99', 'rows': len(preferred), 'users': preferred['user_id'].nunique()},
])

trim_summary = {
    'D_mu_1pct': D_mu_lo,
    'D_mu_99pct': D_mu_hi,
    'D_sigma_1pct': D_sigma_lo,
    'D_sigma_99pct': D_sigma_hi,
    'rows_all_test': int(len(reg)),
    'rows_no_tau_bound': int(len(no_bound)),
    'rows_preferred': int(len(preferred)),
    'rows_dropped_by_tau_bound': int(len(reg) - len(no_bound)),
    'rows_dropped_by_D_trim_after_no_bound': int(len(no_bound) - len(preferred)),
}

print('Treatment trim bounds computed in no-bound sample:')
print(json.dumps({k: v for k, v in trim_summary.items() if 'pct' in k}, indent=2))
display(sample_summary)


Treatment trim bounds computed in no-bound sample:
{
  "D_mu_1pct": -0.02110174376753266,
  "D_mu_99pct": 0.1170824512321263,
  "D_sigma_1pct": -0.09602536518809282,
  "D_sigma_99pct": 0.013100231802274502
}


,sample,rows,users
0,all_test,4069,356
1,no_tau_bound,4062,356
2,preferred_no_bound_trim_D_1_99,3918,356


## 3. Estimate Final Regression

The preferred estimate is based on the no-bound, treatment-trimmed sample. The table also reports two compact robustness checks: the full test set and the no-bound sample before treatment trimming.


In [4]:
def fit_clustered_ols(frame, outcome_col='delta_tau_resid', d_mu_col='D_mu', d_sigma_col='D_sigma'):
    data = frame[['user_id', outcome_col, d_mu_col, d_sigma_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    y = data[outcome_col].astype(float)
    X = sm.add_constant(data[[d_mu_col, d_sigma_col]].astype(float), has_constant='add')
    ols = sm.OLS(y, X).fit()
    clustered = ols.get_robustcov_results(cov_type='cluster', groups=data['user_id'])
    rows = []
    for idx, term in enumerate(['const', 'D_mu', 'D_sigma']):
        rows.append({
            'term': term,
            'coef': float(clustered.params[idx]),
            'cluster_se': float(clustered.bse[idx]),
            't': float(clustered.tvalues[idx]),
            'p_value': float(clustered.pvalues[idx]),
        })
    return {
        'n': int(len(data)),
        'n_users': int(data['user_id'].nunique()),
        'r2': float(ols.rsquared),
        'corr_D_mu_D_sigma': float(data[d_mu_col].corr(data[d_sigma_col])),
        'table': pd.DataFrame(rows),
    }


def summarize_spec(name, frame, note, outcome_col='delta_tau_resid'):
    fit = fit_clustered_ols(frame, outcome_col=outcome_col)
    tab = fit['table'].copy()
    tab.insert(0, 'spec', name)
    tab.insert(1, 'n', fit['n'])
    tab.insert(2, 'n_users', fit['n_users'])
    tab.insert(3, 'r2', fit['r2'])
    tab.insert(4, 'corr_D_mu_D_sigma', fit['corr_D_mu_D_sigma'])
    tab['outcome'] = outcome_col
    tab['note'] = note
    return tab

spec_tables = [
    summarize_spec('all_test', reg, 'Full held-out test sample.'),
    summarize_spec('no_tau_bound', no_bound, 'Drops transitions where tau_t or tau_t1 is at optimizer bound.'),
    summarize_spec('preferred', preferred, 'No tau-bound transitions; D_mu and D_sigma trimmed at 1st/99th percentiles.'),
]
results_df = pd.concat(spec_tables, ignore_index=True)
report_table = results_df[results_df['term'].isin(['D_mu', 'D_sigma'])].copy()

display(report_table[['spec', 'n', 'n_users', 'term', 'coef', 'cluster_se', 't', 'p_value', 'r2']])


,spec,n,n_users,term,coef,cluster_se,t,p_value,r2
1,all_test,4069,356,D_mu,7.771875,4.222818,1.840447,0.066537,0.002436
2,all_test,4069,356,D_sigma,11.688521,5.995035,1.949700,0.051999,0.002436
4,no_tau_bound,4062,356,D_mu,8.088344,4.225789,1.914043,0.056418,0.002586
5,no_tau_bound,4062,356,D_sigma,11.982212,5.997136,1.997989,0.046480,0.002586
7,preferred,3918,356,D_mu,11.742392,4.749162,2.472519,0.013884,0.004121
8,preferred,3918,356,D_sigma,18.177189,6.617437,2.746862,0.006324,0.004121


## 4. Outcome-Winsorized Robustness

This robustness check keeps the same preferred sample but winsorizes the outcome at the full-test 1st and 99th percentiles. It is not the main estimate; it checks whether extreme residualized tau changes drive the result.


In [5]:
out_lo, out_hi = reg['delta_tau_resid'].quantile([0.01, 0.99])
preferred_winsor = preferred.copy()
preferred_winsor['delta_tau_resid_w01_99'] = preferred_winsor['delta_tau_resid'].clip(out_lo, out_hi)

winsor_df = summarize_spec(
    'preferred_outcome_winsor_1_99',
    preferred_winsor,
    f'Preferred sample; outcome winsorized at full-test 1st/99th percentiles [{out_lo:.4f}, {out_hi:.4f}].',
    outcome_col='delta_tau_resid_w01_99',
)

robust_table = winsor_df[winsor_df['term'].isin(['D_mu', 'D_sigma'])].copy()
display(robust_table[['spec', 'n', 'n_users', 'term', 'coef', 'cluster_se', 't', 'p_value', 'r2']])


,spec,n,n_users,term,coef,cluster_se,t,p_value,r2
1,preferred_outcome_winsor_1_99,3918,356,D_mu,11.634313,4.696868,2.477036,0.013713,0.004314
2,preferred_outcome_winsor_1_99,3918,356,D_sigma,18.320771,6.491511,2.822266,0.005037,0.004314


## 5. Save Report Tables

The CSV outputs are designed to be directly usable in the paper draft or appendix.


In [6]:
all_results = pd.concat([results_df, winsor_df], ignore_index=True)
final_report_table = all_results[all_results['term'].isin(['D_mu', 'D_sigma'])].copy()

all_results.to_csv(RESULTS_CSV, index=False)
final_report_table.to_csv(REPORT_TABLE_CSV, index=False)

payload = {
    'main_regression': 'delta_tau_resid = alpha + beta_mu * D_mu + beta_sigma * D_sigma + epsilon',
    'standard_errors': 'clustered by user_id',
    'preferred_spec': 'preferred',
    'preferred_sample': 'no tau-bound transitions; D_mu and D_sigma trimmed at no-bound 1st/99th percentiles',
    'outcome_trimming_main_result': False,
    'trim_summary': trim_summary,
    'results_csv': str(RESULTS_CSV),
    'report_table_csv': str(REPORT_TABLE_CSV),
}
RESULTS_JSON.write_text(json.dumps(payload, indent=2) + '\n')

preferred_terms = final_report_table[final_report_table['spec'].eq('preferred')][['term', 'coef', 'cluster_se', 't', 'p_value']]
print('Preferred final DML result')
print(preferred_terms.to_string(index=False))
print('\nSaved:', RESULTS_CSV)
print('Saved:', REPORT_TABLE_CSV)
print('Saved:', RESULTS_JSON)


Preferred final DML result
   term      coef  cluster_se        t  p_value
   D_mu 11.742392    4.749162 2.472519 0.013884
D_sigma 18.177189    6.617437 2.746862 0.006324

Saved: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_final_dml_regression_results.csv
Saved: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_final_dml_report_table.csv
Saved: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_final_dml_regression_results.json
